# CEG-WM Content V5 — formal initial user-only GPU handoff

This Colab notebook executes the frozen initial invocation only for `content-v5-c5a0c4bf7d6d-805bc21e173a` from `stage-a-content-v5-lf-or-hf@ed5ed5a303c4eb3a8e7cafb8410109b68f5e41fa`. It mounts Drive first, proves the fresh exact checkout, installs only that checkout, validates the public protocol and whitening identities, reads both Secrets, and invokes the existing formal runner exactly once.

The dual-cohort order and all decision logic remain inside the frozen V5 runner; this notebook performs no manifest iteration, cohort loop, Gate calculation, result interpretation, retry, or resume.

Create Colab Secrets named `CEG_WM_ROOT_KEY` and `HF_TOKEN`. Run the cells once from top to bottom. Stop after any failure or interruption.

## 1. Mount Drive, then prove the fresh execution checkout

Drive mounting is the first external action. The source path and exact-bound local and Drive run destinations must all be absent before checkout or installation.

In [ ]:
from google.colab import drive
try:
    drive.mount("/content/drive")
except BaseException:
    print('CEGWM_CONTENT_V5_FORMAL_HANDOFF_FAILURE {"error_class":"OtherOperationalError","execution_exact":"ed5ed5a303c4eb3a8e7cafb8410109b68f5e41fa","run_id":"content-v5-c5a0c4bf7d6d-805bc21e173a","stage":"drive_mount","status":"operational_failure"}', flush=True)
    HANDOFF_FAILED = True
else:
    HANDOFF_FAILED = False

import json
import pathlib
import subprocess
import sys

REPO_URL = "https://github.com/RICHAAARC/CEG-WM.git"
BRANCH = "Content-V5"
EXACT = "ed5ed5a303c4eb3a8e7cafb8410109b68f5e41fa"
RUNNER_MODULE = "experiments.run_content_v5_clean"
FAILURE_PREFIX = "CEGWM_CONTENT_V5_FORMAL_HANDOFF_FAILURE"
ARTIFACT_PREFIX = "CEGWM_CONTENT_V5_FORMAL_ARTIFACT"
METHOD_ID = "content_v5_clean_null_whitened_lf_adaptive_hf_branchwise_or_v1"
CANDIDATE_ID = "content_v5_clean_null_whitened_lf_adaptive_hf_branchwise_or_gate_v1"
PROTOCOL_ID = "cegwm-stage-a-content-v5-whitened-lf-adaptive-hf-branchwise-or-clean-v1"
PROTOCOL_DIGEST = "c5a0c4bf7d6d3521ae233756ea07753dd002d842662b50f82a86de6a0f96c204"
STATE_SCHEMA_ID = "content_v5_umbrella_whole_unit_checkpoint_state_v1"
ARTIFACT_CONTRACT_ID = "content_v5_reference_then_primary_umbrella_artifact_v1"
RUN_ID = "content-v5-c5a0c4bf7d6d-805bc21e173a"
PUBLIC_KEY_DIGEST = "805bc21e173a83898f3b7034d75e6ed02f65894a6885377d9659ee3091b4dd77"
PRIMARY_ROSTER_SHA256 = "5303a0284e36d2e6e159526c7ba61a7106fb3db72de35f0ada98fcfd5da2ec2c"
CONTROL_ROSTER_SHA256 = "dd30c719ae5a48b2a9a652420a3237adb74ffd26af8bac90e25c1d03fe845b88"
MODEL_ID = "stabilityai/stable-diffusion-3.5-medium"
DINO_ASSET_ID = "facebook/dinov2-small"
WHITENING_ASSET_SHA256 = "a7021dd8b98bc4282b98ed5d1fe276236d99a3c9e80b9bdce015d28cf715633f"
WHITENING_SIDECAR_FILE_SHA256 = "c900cce0980348eeadcf07d782b6169c4d46ac55d7154db0fc0a0a878cce0ced"

repo = pathlib.Path("/content/cegwm-stage-a-content-v5-lf-or-hf-source")
local_work_root = pathlib.Path("/content/cegwm-stage-a-content-v5-lf-or-hf-local")
artifact_sink = pathlib.Path("/content/drive/MyDrive/CEG-WM/stage_a_content_v5_lf_or_hf")
bound_local_run = local_work_root / RUN_ID
bound_drive_run = artifact_sink / RUN_ID
RUNNER_ATTEMPTED = False

_ALLOWED_ERRORS = {
    "CalledProcessError", "FileExistsError", "ImportError",
    "ModuleNotFoundError", "OSError", "RuntimeError",
    "TypeError", "UnicodeDecodeError", "ValueError",
}

def fail(stage, error_class="RuntimeError"):
    global HANDOFF_FAILED
    if HANDOFF_FAILED:
        return
    if error_class not in _ALLOWED_ERRORS:
        error_class = "OtherOperationalError"
    HANDOFF_FAILED = True
    payload = {
        "status": "operational_failure",
        "run_id": RUN_ID,
        "execution_exact": EXACT,
        "stage": stage,
        "error_class": error_class,
    }
    line = FAILURE_PREFIX + " " + json.dumps(payload, sort_keys=True, separators=(",", ":"))
    if len(line.encode("utf-8")) <= 4096:
        print(line, flush=True)

def git(*args):
    return subprocess.run(
        ["git", *args], cwd=repo, check=True,
        capture_output=True, text=True,
    ).stdout.strip()

if not HANDOFF_FAILED:
    try:
        if repo.exists() or bound_local_run.exists() or bound_drive_run.exists():
            raise FileExistsError
        subprocess.run(
            ["git", "clone", "--single-branch", "--branch", BRANCH, REPO_URL, str(repo)],
            check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
        )
        if (
            git("branch", "--show-current") != BRANCH
            or git("rev-parse", "HEAD") != EXACT
            or git("status", "--porcelain") != ""
            or bound_local_run.exists()
            or bound_drive_run.exists()
        ):
            raise RuntimeError
    except BaseException as error:
        fail("source_checkout_identity_validation", type(error).__name__)

## 2. Install only the checked-out project

This installs only the project declared by the frozen checkout, then rechecks the named branch, exact revision, clean state, and initial-only destinations.

In [ ]:
if not HANDOFF_FAILED:
    try:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", str(repo)],
            check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
        )
        if (
            git("branch", "--show-current") != BRANCH
            or git("rev-parse", "HEAD") != EXACT
            or git("status", "--porcelain") != ""
            or bound_local_run.exists()
            or bound_drive_run.exists()
        ):
            raise RuntimeError
    except BaseException as error:
        fail("dependency_install_and_source_recheck", type(error).__name__)

## 3. Validate public identities and invoke the formal runner once

The root key and Hugging Face token enter only the child environment and are cleared promptly. Raw child stdout and stderr are never printed. The formal runner remains the sole writer of checkpoint and terminal artifact pairs.

In [ ]:
import hashlib
import os

if not HANDOFF_FAILED and not RUNNER_ATTEMPTED:
    RUNNER_ATTEMPTED = True
    process = None
    runner_env = None
    root_key = ""
    hf_token = ""
    normalized_key = b""
    captured = bytearray()
    capture_overflow = False
    runner_rc = None
    launch_error = None
    CAPTURE_LIMIT = 8192
    try:
        from google.colab import userdata
        from cegwm.method.content_whitening_v4 import (
            ASSET_REPO_PATH,
            ASSET_SIDECAR_REPO_PATH,
            load_frozen_content_v4_whitening_asset,
        )
        from cegwm.protocol.content_chain_v5 import load_content_v5_clean_protocol
        from cegwm.shared.keys import normalize_detection_key, public_key_digest

        if (
            git("branch", "--show-current") != BRANCH
            or git("rev-parse", "HEAD") != EXACT
            or git("status", "--porcelain") != ""
            or bound_local_run.exists()
            or bound_drive_run.exists()
        ):
            raise RuntimeError

        config_root = repo / "configs" / "content_chain"
        config_path = config_root / "content_v5_lf_or_hf_clean_v1.json"
        primary_roster_path = config_root / "content_v5_primary_evaluation_v1.jsonl"
        control_roster_path = config_root / "content_adaptive_dual_branch_v2_clean.jsonl"
        protocol = load_content_v5_clean_protocol(
            config_path, primary_roster_path, control_roster_path,
        )
        load_frozen_content_v4_whitening_asset(repo)
        asset_path = repo / ASSET_REPO_PATH
        asset_sidecar_path = repo / ASSET_SIDECAR_REPO_PATH
        method_identities = protocol.config["method_identities"]
        generation_runtime = protocol.config["generation_runtime"]
        content_analysis = protocol.config["content_analysis"]
        lf_operator = protocol.config["lf_detection_operator"]
        execution_flow = protocol.config["execution_flow"]
        if (
            protocol.protocol_id != PROTOCOL_ID
            or protocol.protocol_digest != PROTOCOL_DIGEST
            or len(protocol.cohorts["control_1"]) != 8
            or len(protocol.cohorts["primary_1"]) != 8
            or hashlib.sha256(primary_roster_path.read_bytes()).hexdigest() != PRIMARY_ROSTER_SHA256
            or hashlib.sha256(control_roster_path.read_bytes()).hexdigest() != CONTROL_ROSTER_SHA256
            or execution_flow["cohorts_in_order"][0]["cohort_id"] != "control_1"
            or execution_flow["cohorts_in_order"][1]["cohort_id"] != "primary_1"
            or execution_flow["umbrella_state_schema_id"] != STATE_SCHEMA_ID
            or execution_flow["umbrella_artifact_contract_id"] != ARTIFACT_CONTRACT_ID
            or method_identities["content_method_id"] != METHOD_ID
            or method_identities["evaluated_candidate_id"] != CANDIDATE_ID
            or generation_runtime["model_id"] != MODEL_ID
            or content_analysis["asset_id"] != DINO_ASSET_ID
            or lf_operator["asset_sha256"] != WHITENING_ASSET_SHA256
            or lf_operator["asset_sidecar_sha256"] != WHITENING_SIDECAR_FILE_SHA256
            or hashlib.sha256(asset_path.read_bytes()).hexdigest() != WHITENING_ASSET_SHA256
            or hashlib.sha256(asset_sidecar_path.read_bytes()).hexdigest() != WHITENING_SIDECAR_FILE_SHA256
        ):
            raise RuntimeError

        root_key = userdata.get("CEG_WM_ROOT_KEY")
        hf_token = userdata.get("HF_TOKEN")
        if not isinstance(root_key, str) or not root_key.strip():
            raise RuntimeError
        if not isinstance(hf_token, str) or not hf_token.strip():
            raise RuntimeError
        normalized_key = normalize_detection_key(root_key)
        key_digest = public_key_digest(normalized_key)
        normalized_key = b""
        if (
            key_digest != PUBLIC_KEY_DIGEST
            or RUN_ID != "content-v5-" + protocol.protocol_digest[:12] + "-" + key_digest[:12]
        ):
            raise RuntimeError

        secret_markers = ("TOKEN", "KEY", "SECRET", "PASSWORD", "CREDENTIAL")
        runner_env = {
            name: value for name, value in os.environ.items()
            if not any(marker in name.upper() for marker in secret_markers)
        }
        runner_env["CEG_WM_ROOT_KEY"] = root_key
        runner_env["HF_TOKEN"] = hf_token
        root_key = ""
        hf_token = ""
        process = subprocess.Popen(
            [
                sys.executable, "-m", RUNNER_MODULE,
                "--repo-root", str(repo),
                "--expected-exact", EXACT,
                "--local-work-root", str(local_work_root),
                "--artifact-sink", str(artifact_sink),
            ],
            cwd=repo, env=runner_env, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL,
        )
        while True:
            chunk = process.stdout.read(1024)
            if not chunk:
                break
            remaining = max(0, CAPTURE_LIMIT - len(captured))
            captured.extend(chunk[:remaining])
            if len(chunk) > remaining:
                capture_overflow = True
        runner_rc = process.wait()
    except BaseException as error:
        launch_error = type(error).__name__
        if process is not None and process.poll() is None:
            process.kill()
            process.wait()
    finally:
        root_key = ""
        hf_token = ""
        normalized_key = b""
        if runner_env is not None:
            runner_env.pop("CEG_WM_ROOT_KEY", None)
            runner_env.pop("HF_TOKEN", None)
        runner_env = None

    captured.clear()
    if launch_error is not None:
        fail("formal_runner_launch", launch_error)
    elif runner_rc != 0:
        fail("formal_runner_nonzero")
    elif capture_overflow:
        fail("formal_runner_stdout_overflow")

## 4. Return existing Drive artifact pairs

This cell is runner-free and read-only. It discovers the existing terminal pair, or every complete contiguous checkpoint pair when no terminal pair exists, and validates each sidecar's exact ZIP filename binding before printing one bounded path-and-declared-hash receipt.

In [ ]:
import json
import pathlib
import re
from google.colab import drive

RUN_ID = "content-v5-c5a0c4bf7d6d-805bc21e173a"
ARTIFACT_PREFIX = "CEGWM_CONTENT_V5_FORMAL_ARTIFACT"
ARTIFACT_FAILURE_PREFIX = "CEGWM_CONTENT_V5_FORMAL_HANDOFF_FAILURE"
artifact_sink = pathlib.Path("/content/drive/MyDrive/CEG-WM/stage_a_content_v5_lf_or_hf")
run_dir = artifact_sink / RUN_ID
prior_failure = bool(globals().get("HANDOFF_FAILED", False))
artifact_failed = False

def artifact_fail(stage):
    global artifact_failed
    if artifact_failed:
        return
    artifact_failed = True
    if not prior_failure:
        payload = {
            "status": "operational_failure",
            "run_id": RUN_ID,
            "stage": stage,
            "error_class": "RuntimeError",
        }
        line = ARTIFACT_FAILURE_PREFIX + " " + json.dumps(payload, sort_keys=True, separators=(",", ":"))
        if len(line.encode("utf-8")) <= 4096:
            print(line, flush=True)

def validate_pair(archive_path, sidecar_path):
    if not archive_path.is_file() or not sidecar_path.is_file():
        raise RuntimeError
    sidecar_text = sidecar_path.read_text(encoding="ascii")
    match = re.fullmatch(r"([0-9a-f]{64})  ([^\s]+)\n", sidecar_text)
    if match is None or match.group(2) != archive_path.name:
        raise RuntimeError
    return {
        "archive_path": str(archive_path),
        "sidecar_path": str(sidecar_path),
        "sha256": match.group(1),
    }

if not prior_failure and not pathlib.Path("/content/drive/MyDrive").exists():
    try:
        drive.mount("/content/drive")
    except BaseException:
        artifact_fail("drive_mount")

if not artifact_failed and pathlib.Path("/content/drive/MyDrive").exists():
    try:
        terminal_archive = run_dir / (RUN_ID + ".zip")
        terminal_sidecar = run_dir / (RUN_ID + ".zip.sha256")
        terminal_presence = (terminal_archive.exists(), terminal_sidecar.exists())
        if terminal_presence == (True, True):
            artifact_kind = "terminal"
            pairs = [validate_pair(terminal_archive, terminal_sidecar)]
        elif terminal_presence != (False, False):
            raise RuntimeError
        else:
            checkpoint_archives = sorted(run_dir.glob(RUN_ID + ".checkpoint-*.zip"))
            checkpoint_sidecars = sorted(run_dir.glob(RUN_ID + ".checkpoint-*.zip.sha256"))
            if not checkpoint_archives:
                raise RuntimeError
            sequence_to_archive = {}
            for archive_path in checkpoint_archives:
                match = re.fullmatch(re.escape(RUN_ID) + r"\.checkpoint-([0-9]{4})\.zip", archive_path.name)
                if match is None:
                    raise RuntimeError
                sequence_to_archive[int(match.group(1))] = archive_path
            if sorted(sequence_to_archive) != list(range(len(sequence_to_archive))):
                raise RuntimeError
            expected_sidecars = {
                pathlib.Path(str(archive_path) + ".sha256")
                for archive_path in sequence_to_archive.values()
            }
            if set(checkpoint_sidecars) != expected_sidecars:
                raise RuntimeError
            artifact_kind = "checkpoint"
            pairs = [
                validate_pair(sequence_to_archive[index], pathlib.Path(str(sequence_to_archive[index]) + ".sha256"))
                for index in range(len(sequence_to_archive))
            ]
        receipt = {
            "status": "drive_artifacts_ready",
            "run_id": RUN_ID,
            "artifact_kind": artifact_kind,
            "pairs": pairs,
        }
        receipt_line = ARTIFACT_PREFIX + " " + json.dumps(receipt, sort_keys=True, separators=(",", ":"))
        if len(receipt_line.encode("utf-8")) > 4096:
            raise RuntimeError
        print(receipt_line, flush=True)
    except BaseException:
        artifact_fail("artifact_pair_validation")

## Stop boundary

Return only the bounded Drive artifact receipt and the referenced existing ZIP/SHA-256 pairs, or the single sanitized failure line when no valid pair is available. The notebook does not relaunch the runner, mutate artifacts, expose child streams, reveal Secrets or private state, or interpret Gates and scientific outcomes.